## Part A: Regular expressions (4 points)

In [1]:
import re
import nltk
nltk.download('gutenberg')
nltk.download('punkt_tab')
from nltk.corpus import gutenberg

[nltk_data] Downloading package gutenberg to /Users/sakis/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.
[nltk_data] Downloading package punkt_tab to /Users/sakis/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Complete the following function using the search function of the re library to find and return the year that a file in the gutenberg collection was written. Return the year as string or the empty string if not found. (2 points)

In [21]:
def get_year(file):
    contents = gutenberg.raw(file)
    re_year = r"\d{4}"
    rex = re.compile(re_year)
    res = rex.search(contents)
    
    if not res:
        return ""
    return res[0]

In [20]:
"""Testing that the function returns the correct results"""
assert get_year('carroll-alice.txt') == '1865'
assert get_year('shakespeare-hamlet.txt') == '1599'
assert get_year('milton-paradise.txt') == '1667'
assert get_year('melville-moby_dick.txt') == '1851'

Complete the following function using the findall function of the re library to compute and report the number of times some text appears in a file of the gutenberg collection. (1 point)

In [26]:
def find_mentions(text, file):
    contents = gutenberg.raw(file)
    needle = rf"{text}"
    rex = re.compile(needle)
    res = rex.findall(contents)
    return len(res)

12


12

In [27]:
"""Testing that the function returns the correct results"""
assert find_mentions('CHAPTER ', 'carroll-alice.txt') == 12
assert find_mentions('UNK', 'shakespeare-caesar.txt') == 0
assert find_mentions('Caesar', 'shakespeare-caesar.txt') == 227
assert find_mentions('girl', 'austen-emma.txt') == 60

12
0
227
60


Complete the following function to return the mean number of words in the sentences of a gutenberg file that is given as input. The number should be rounded to 2 decimals. (1 point)

In [43]:
def average_sentence_length(file):
    sentences = gutenberg.sents(file)
    total_tokens = sum([len(sentence) for sentence in sentences])
    return round(total_tokens/len(sentences), 2)

In [37]:
"""Testing that the function returns the correct results"""
assert average_sentence_length('shakespeare-caesar.txt') == 11.94
assert average_sentence_length('austen-emma.txt') == 24.82
assert average_sentence_length('milton-paradise.txt') == 52.31
assert average_sentence_length('melville-moby_dick.txt') == 25.93

AssertionError: 

## Part B: Byte-Pair Encoding (6 points)

Implement a function that returns the vocabulary learned by the byte-pair encoding algorithm given a corpus, after a number of iterations. The characters in the corpus, sorted in alphabetical order, will be the first members of the vocabulary, to be extended with as many tokens as the number of iterations given.

In [71]:
from nltk import word_tokenize

def get_vocabulary(words: list(str)) -> dict:
    vocab = {}
    for word in words:
        for c in word:
            if c not in vocab:
                vocab[c] = 0
            vocab[c] += 1

    return vocab

def get_vocabularized_words(words: list, vocabulary: dict) -> list:
    ordered_vocabulary = sorted(vocabulary.keys(), key=lambda x: vocabulary[x])
    print(f"ordered vocab: {ordered_vocabulary}")
    vocabed_words = []
    for word in words:
        vocabed_word = []
        for t in ordered_vocabulary:
            if t in word:
                vocabed_word.append(t)
                word.replace(t, "")
        vocabed_words.append(vocabed_word)

    return vocabed_words
            

def bpe_train(text, iterations):
    # prefix all spaces with underscore, to help with the vocabulary
    text = text.replace(" ", " _")
    words = word_tokenize(text)
    # just to make sure word_tokenize didn't split the underscores from the words
    assert len(words) == len(text.split())
    
    vocabulary = get_vocabulary(words)
    vocabularized_words = get_vocabularized_words(words, vocabulary)
    print(vocabularized_words)
    tmp_vocabulary = {}
    for word in words:
        for i, char in enumerate(word):
            if i + 1 >= len(word):
                break
            comb = f"{word[i]}{word[i+1]}"
            if comb not in tmp_vocabulary:
                tmp_vocabulary[comb] = 0
            tmp_vocabulary[comb] += 1

    print(list("test"))
    print(tmp_vocabulary)
    print(vocabulary)
    

bpe_train("set new new renew reset renew", 4)

ordered vocab: ['s', 't', 'r', 'n', 'w', '_', 'e']
[['s', 't', 'e'], ['n', 'w', '_', 'e'], ['n', 'w', '_', 'e'], ['r', 'n', 'w', '_', 'e'], ['s', 't', 'r', '_', 'e'], ['r', 'n', 'w', '_', 'e']]
['t', 'e', 's', 't']
{'se': 2, 'et': 2, '_n': 2, 'ne': 4, 'ew': 4, '_r': 3, 're': 3, 'en': 2, 'es': 1}
{'s': 2, 'e': 9, 't': 2, '_': 5, 'n': 4, 'w': 4, 'r': 3}


In [49]:
"""Testing that the function returns the correct results"""
corpus = "set new new renew reset renew"
assert set(bpe_train(corpus, 4)) == set(['_', 'e', 'n', 'r', 's', 't', 'w', 'ne', 'new', '_r', '_re'])
assert set(bpe_train(corpus, 8)) == set(['_', 'e', 'n', 'r', 's', 't', 'w', 'ne', 'new', '_r', '_re', 'se', 'set', '_new', '_renew'])
assert set(bpe_train(corpus, 9)) == set(['_', 'e', 'n', 'r', 's', 't', 'w', 'ne', 'new', '_r', '_re', 'se', 'set', '_new', '_renew', '_reset']);

corpus = "hello world my large language model is coming after you beware"
assert set(bpe_train(corpus, 6)) == set(['_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'r', 's', 't', 'u', 'w', 'y', 'el', '_m', '_l', '_la', 'ge', 'ng'])

corpus = "low low low low low lowest lowest newer newer newer newer newer newer wider wider wider new new"
assert set(bpe_train(corpus, 4)) == set(['_', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w', 'er', '_n', '_ne', '_new'])

['set', 'new', 'new', 'renew', 'reset', 'renew']


TypeError: 'NoneType' object is not iterable